# 01 - Data Preparation (HF Alzheimer Dataset)

Objectif:
- Charger la dataset Hugging Face `DS23-KI-Projekt/alzheimerdataset_split`.
- Construire un fichier NPZ compatible avec le pipeline MIA existant.

Note:
- Si la dataset est privee, execute d'abord `huggingface-cli login`.

In [1]:
import os
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

dataset_name = "DS23-KI-Projekt/alzheimerdataset_split"
print(f'Loading dataset: {dataset_name}')

token = os.getenv('HF_TOKEN', None)
if token:
    ds = load_dataset(dataset_name, token=token)
else:
    ds = load_dataset(dataset_name)

print(ds)
print('Splits:', list(ds.keys()))

c:\Users\khalil\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset: DS23-KI-Projekt/alzheimerdataset_split
DatasetDict({
    train: Dataset({
        features: ['Age', 'BMI', 'Education Level', 'Gender', 'Family History of Alzheimer’s', 'Cognitive Test Score', 'Genetic Risk Factor (APOE-ε4 allele)', 'Alzheimer’s Diagnosis'],
        num_rows: 59401
    })
    validation: Dataset({
        features: ['Age', 'BMI', 'Education Level', 'Gender', 'Family History of Alzheimer’s', 'Cognitive Test Score', 'Genetic Risk Factor (APOE-ε4 allele)', 'Alzheimer’s Diagnosis'],
        num_rows: 14851
    })
})
Splits: ['train', 'validation']


In [2]:
import re

def split_to_df(split):
    try:
        return split.to_pandas()
    except Exception:
        return pd.DataFrame(split)

split_dfs = {name: split_to_df(ds[name]) for name in ds.keys()}
for name, df in split_dfs.items():
    print(name, df.shape)

all_cols = set()
for df in split_dfs.values():
    all_cols.update(df.columns.tolist())

def _norm_col(c):
    # Normalize unicode punctuation and keep only alphanumerics for robust matching.
    c = str(c).lower()
    c = c.replace("’", "'")
    c = re.sub(r"[^a-z0-9]+", " ", c)
    return c.strip()

label_col = None
norm_map = {col: _norm_col(col) for col in all_cols}

# First pass: strongly prefer explicit diagnosis-like target columns.
priority_phrases = [
    "alzheimer diagnosis",
    "alzheimers diagnosis",
    "diagnosis",
]
for phrase in priority_phrases:
    matches = [col for col, ncol in norm_map.items() if ncol == phrase]
    if matches:
        label_col = sorted(matches)[0]
        break

# Second pass: exact/common generic label names.
if label_col is None:
    generic_keywords = ["label", "labels", "target", "class", "y"]
    for col, ncol in norm_map.items():
        if ncol in generic_keywords:
            label_col = col
            break

# Third pass: fuzzy fallback only if still not found.
if label_col is None:
    for col, ncol in norm_map.items():
        if "diagnosis" in ncol:
            label_col = col
            break

if label_col is None:
    raise ValueError(f'No label column found. Available columns: {sorted(list(all_cols))}')

print('Selected label column:', label_col)

df_all = pd.concat(list(split_dfs.values()), axis=0, ignore_index=True)

if df_all[label_col].dtype == 'O':
    y_all = pd.Categorical(df_all[label_col]).codes.astype(np.int32)
else:
    y_all = df_all[label_col].astype(np.int32).to_numpy()

feature_df = df_all.drop(columns=[label_col])

# Keep simple tabular representation: numeric columns + one-hot for categorical columns.
feature_df = pd.get_dummies(feature_df, drop_first=True)
feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

X_all = feature_df.to_numpy(dtype=np.float32)
feature_names = feature_df.columns.to_numpy(dtype=object)

print('Final feature matrix:', X_all.shape)
print('Class distribution:', pd.Series(y_all).value_counts().to_dict())

# Split target population vs shadow pool
X_target, X_shadow_raw, y_target, y_shadow = train_test_split(
    X_all, y_all,
    test_size=0.30,
    random_state=SEED,
    stratify=y_all
)

# Split target train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_target, y_target,
    test_size=0.75,
    random_state=SEED + 1,
    stratify=y_target
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc = scaler.transform(X_test).astype(np.float32)
X_shadow_sc = scaler.transform(X_shadow_raw).astype(np.float32)
X_all_sc = scaler.transform(X_all).astype(np.float32)

X_train_seq = X_train_sc.reshape(-1, 1, X_train_sc.shape[1])
X_test_seq = X_test_sc.reshape(-1, 1, X_test_sc.shape[1])

out_path = 'prepared_alzheimer_hf_transformer.npz'
np.savez(
    out_path,
    X=X_all_sc,
    y=y_all.astype(np.int32),
    X_train=X_train_sc,
    X_test=X_test_sc,
    y_train=y_train.astype(np.int32),
    y_test=y_test.astype(np.int32),
    X_shadow_raw=X_shadow_sc,
    X_shadow=X_shadow_sc,
    y_shadow=y_shadow.astype(np.int32),
    X_train_seq=X_train_seq,
    X_test_seq=X_test_seq,
    seed=np.array([SEED], dtype=np.int32),
    feature_names=feature_names,
    label_name=np.array([label_col], dtype=object),
)

print('Saved:', out_path)
print('X_train:', X_train_sc.shape, 'X_test:', X_test_sc.shape, 'X_shadow:', X_shadow_sc.shape)

train (59401, 8)
validation (14851, 8)
Selected label column: Alzheimer’s Diagnosis
Final feature matrix: (74252, 7)
Class distribution: {0: 43552, 1: 30700}
Saved: prepared_alzheimer_hf_transformer.npz
X_train: (12994, 7) X_test: (38982, 7) X_shadow: (22276, 7)
